# Correct ANTs Warps — Multi-Constraint Comparison

Same synthetic 3D warp as `correct-ants-warps.ipynb`, but each slice is
corrected under four constraint modes and the results are compared using
the injectivity checks from `test-global-folding.ipynb`:

| Mode | `enforce_shoelace` | `enforce_injectivity` |
|------|-------------------|----------------------|
| jdet-only | False | False |
| jdet+shoe | True | False |
| jdet+inject | False | True |
| jdet+shoe+inject | True | True |

For each slice the notebook shows:
1. Jacobian-determinant heat-maps: before correction + after each mode (side-by-side)
2. Per-mode injectivity summary printed to stdout (Jdet, Shoelace, Monotonicity)
3. A final summary table across all slices and modes


## Imports

In [ ]:
import time

import numpy as np
import matplotlib.pyplot as plt

from dvfopt import (
    jacobian_det2D,
    iterative_serial,
    shoelace_det2D,
)
from dvfopt.jacobian import _monotonicity_diffs_2d


## Configuration

In [ ]:
JDET_THRESHOLD = 0.01

MODES = [
    ("jdet-only",         dict(enforce_shoelace=False, enforce_injectivity=False)),
    ("jdet+shoe",         dict(enforce_shoelace=True,  enforce_injectivity=False)),
    ("jdet+inject",       dict(enforce_shoelace=False, enforce_injectivity=True)),
    ("jdet+shoe+inject",  dict(enforce_shoelace=True,  enforce_injectivity=True)),
]

SLICE_LABELS = [
    "crossing blobs", "tanh pinch", "sine fold",
    "radial implosion", "y-squeeze", "local push",
]


## Generate synthetic warp

In [ ]:
H, W, D = 40, 40, 6
yy, xx = np.mgrid[0:H, 0:W].astype(float)
cy, cx = H / 2.0, W / 2.0

warp_zyx = np.zeros((3, H, W, D), dtype=np.float64)

# Slice 0: Crossing blobs
ra = np.sqrt((yy - H*0.3)**2 + (xx - W*0.3)**2)
rb = np.sqrt((yy - H*0.7)**2 + (xx - W*0.7)**2)
ma = (ra < H*0.15).astype(float)
mb = (rb < H*0.15).astype(float)
warp_zyx[1, :, :, 0] = ma * H*0.4 - mb * H*0.4
warp_zyx[2, :, :, 0] = ma * W*0.4 - mb * W*0.4

# Slice 1: Tanh pinch
warp_zyx[1, :, :, 1] = -(H * 0.15) * np.tanh(0.3 * (yy - cy))

# Slice 2: Sine fold
warp_zyx[2, :, :, 2] = (W * 0.2) * np.sin(4 * np.pi * xx / W)

# Slice 3: Radial implosion
r3 = np.sqrt((yy - cy)**2 + (xx - cx)**2) + 1e-6
angle3 = np.arctan2(yy - cy, xx - cx)
strength3 = np.exp(-r3**2 / (2 * (H*0.2)**2)) * H * 0.7
warp_zyx[1, :, :, 3] = -strength3 * np.sin(angle3)
warp_zyx[2, :, :, 3] = -strength3 * np.cos(angle3)

# Slice 4: Y-squeeze
warp_zyx[1, :, :, 4] = (
    np.where(yy < cy, H*0.25, -H*0.25)
    * np.exp(-((yy - cy) / (H*0.15))**2)
)

# Slice 5: Local push
r5 = np.sqrt((yy - H*0.4)**2 + (xx - W*0.4)**2) + 1e-6
strength5 = np.exp(-r5**2 / (2*(H*0.08)**2)) * H * 0.8
warp_zyx[1, :, :, 5] = strength5 * (yy - H*0.4) / r5
warp_zyx[2, :, :, 5] = strength5 * (xx - W*0.4) / r5


print(f"Generated synthetic warp: (3, {H}, {W}, {D})")
for z in range(D):
    dy_s = warp_zyx[1, :, :, z]
    dx_s = warp_zyx[2, :, :, z]
    jac_z = jacobian_det2D(np.stack([dy_s, dx_s]))
    n_neg = int((jac_z <= 0).sum())
    print(f"  slice {z} ({SLICE_LABELS[z]}):  neg_jdet={n_neg}  min={float(jac_z.min()):+.4f}")


## ANTs warp convention

ANTs displacement fields are **pull-back (backward) maps**: for every voxel
position in **fixed space** the warp gives the displacement to the corresponding
location in **moving space**.  This is identical to the dvfopt convention
(`[dz, dy, dx]` in fixed-space coordinates, pull-back semantics).

The synthetic slices below follow the same convention: each slice shows how
fixed-space grid nodes are displaced when sampling the moving image.


In [ ]:
# Show the planned warping for each slice as a deformed grid
# (pull-back: fixed-space grid displaced toward moving space)
stride = 2

fig, axes = plt.subplots(2, 3, figsize=(13, 8), constrained_layout=True)

for z, ax in enumerate(axes.flat):
    dy = warp_zyx[1, :, :, z]
    dx = warp_zyx[2, :, :, z]

    rows = np.arange(0, H, stride)
    cols = np.arange(0, W, stride)
    rr, cc = np.meshgrid(rows, cols, indexing="ij")
    def_y = rr + dy[rr, cc]
    def_x = cc + dx[rr, cc]

    phi_z = np.stack([dy, dx])
    jac_z = np.squeeze(jacobian_det2D(phi_z))
    n_neg = int((jac_z <= 0).sum())
    vmax = max(abs(float(jac_z.min())), float(jac_z.max()), 1.0)
    # No extent: imshow maps array[i,j] -> pixel (x=j, y=i) in integer coords
    ax.imshow(jac_z, cmap="RdBu_r", vmin=-vmax, vmax=vmax,
              alpha=0.4, origin="upper")

    for i in range(def_y.shape[0]):
        ax.plot(def_x[i, :], def_y[i, :], "k-", lw=0.7, alpha=0.85)
    for j in range(def_y.shape[1]):
        ax.plot(def_x[:, j], def_y[:, j], "k-", lw=0.7, alpha=0.85)

    ax.set_xlim(-0.5, W - 0.5)
    ax.set_ylim(H - 0.5, -0.5)
    subtitle = f"neg_jdet={n_neg}  min={float(jac_z.min()):+.3f}"
    ax.set_title(f"Slice {z}: {SLICE_LABELS[z]}" + chr(10) + subtitle, fontsize=9)
    ax.set_aspect("equal")
    ax.axis("off")

plt.suptitle(
    "Planned warping -- deformed grid per slice  |  pull-back: fixed -> moving",
    fontsize=11,
)
plt.show()


## Helpers

In [ ]:
def extract_phi(warp_zyx, z):
    dy = warp_zyx[1, :, :, z].copy()
    dx = warp_zyx[2, :, :, z].copy()
    deformation = np.zeros((3, 1, H, W), dtype=np.float64)
    deformation[1, 0] = dy
    deformation[2, 0] = dx
    return deformation


def injectivity_stats(phi):
    jac  = np.squeeze(jacobian_det2D(phi))
    shoe = np.squeeze(shoelace_det2D(phi))
    h_m, v_m = _monotonicity_diffs_2d(phi[0], phi[1])
    return dict(
        n_neg_jac  = int((jac[1:-1, 1:-1] <= 0).sum()),
        n_neg_shoe = int((shoe[1:-1, 1:-1] <= 0).sum()),
        n_h_viol   = int((h_m[1:-1, 1:-1] <= 0).sum()),
        n_v_viol   = int((v_m[1:-1, 1:-1] <= 0).sum()),
        min_jac    = float(jac.min()),
        jac        = jac,
    )


def show_jdet_map(jac2d, title, ax):
    vmax = max(abs(float(jac2d.min())), abs(float(jac2d.max())), 1.0)
    im = ax.imshow(jac2d, cmap="RdBu_r", vmin=-vmax, vmax=vmax)
    n_neg = int((jac2d <= 0).sum())
    ax.set_title(title + chr(10) + f"neg={n_neg}  min={jac2d.min():+.3f}", fontsize=8)
    ax.axis("off")
    return im


## Run corrections across all modes

In [ ]:
all_results = {}   # {z: {mode_label: stats_dict}}

for z in range(D):
    label = SLICE_LABELS[z]
    deformation = extract_phi(warp_zyx, z)
    phi_init = np.stack([deformation[1, 0], deformation[2, 0]])

    stats_init = injectivity_stats(phi_init)
    n_neg_init = stats_init['n_neg_jac']

    sep = "=" * 72
    print()
    print(sep)
    print(f"  Slice {z}  ({label})  |  {H}x{W}  |  neg_jdet={n_neg_init}  min={stats_init['min_jac']:+.4f}")
    print(sep)

    mode_results = {}

    ncols = 1 + len(MODES)
    fig, axes = plt.subplots(1, ncols, figsize=(3.5 * ncols, 3.2), constrained_layout=True)
    im_ref = show_jdet_map(stats_init['jac'], "Before", axes[0])

    for col, (mode_label, flags) in enumerate(MODES, start=1):
        t0 = time.perf_counter()
        phi_c = iterative_serial(
            deformation.copy(), verbose=0, threshold=JDET_THRESHOLD, **flags
        )
        elapsed = time.perf_counter() - t0

        st = injectivity_stats(phi_c)
        st['time'] = elapsed
        st['phi']  = phi_c
        mode_results[mode_label] = st

        show_jdet_map(st['jac'], mode_label, axes[col])

        ok_j = "OK" if st['n_neg_jac']  == 0 else f"FAIL({st['n_neg_jac']})"
        ok_s = "OK" if st['n_neg_shoe'] == 0 else f"FAIL({st['n_neg_shoe']})"
        ok_m = ("OK" if st['n_h_viol'] == 0 and st['n_v_viol'] == 0
                else f"FAIL(h={st['n_h_viol']},v={st['n_v_viol']})")
        print(f"  [{mode_label:20s}]  jdet={ok_j:12s}  shoe={ok_s:12s}  mono={ok_m}  t={elapsed:.2f}s")

    plt.colorbar(im_ref, ax=axes.tolist(), shrink=0.7, label="Jacobian det")
    plt.suptitle(f"Slice {z} ({label}) — Jdet before & after each mode", fontsize=10)
    plt.show()

    all_results[z] = mode_results


## Summary table

In [ ]:
def tick(n):
    return "  OK  " if n == 0 else f"FAIL({n})"

hdr = (f"{'Sl':>2s}  {'Pattern':<18s}  {'Mode':<20s}  "
       f"{'Jdet':>12s}  {'Shoelace':>10s}  {'Mono-h':>8s}  {'Mono-v':>8s}  {'Time':>6s}")
print(hdr)
print("-" * len(hdr))

for z in range(D):
    for i, (mode_label, _) in enumerate(MODES):
        st = all_results[z][mode_label]
        pat = SLICE_LABELS[z] if i == 0 else ""
        sl  = str(z) if i == 0 else ""
        print(f"{sl:>2s}  {pat:<18s}  {mode_label:<20s}  "
              f"{tick(st['n_neg_jac']):>12s}  {tick(st['n_neg_shoe']):>10s}  "
              f"{tick(st['n_h_viol']):>8s}  {tick(st['n_v_viol']):>8s}  "
              f"{st['time']:>5.2f}s")
    print()
